In [11]:
!pkill -f "ollama serve"
!pip install -q -U transformers accelerate timm fastapi uvicorn pillow librosa soundfile

In [12]:
%%writefile server.py
import os, base64, tempfile, time, uuid, threading, torch
from fastapi import FastAPI, Header, HTTPException, Request
from transformers import AutoProcessor, AutoModelForImageTextToText

API_KEY = os.environ.get("API_KEY", "")
MODEL_ID = "google/gemma-3n-E2B-it"

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="cuda:0").eval()
lock = threading.Lock()
app = FastAPI()

def save_tmp(b64, suffix):
    if b64.startswith("data:"):
        b64 = b64.split(",", 1)[1]
    f = tempfile.NamedTemporaryFile(delete=False, suffix=suffix)
    f.write(base64.b64decode(b64)); f.close()
    return f.name

def convert(messages):
    out = []
    for m in messages:
        c = m["content"]
        if isinstance(c, str):
            out.append({"role": m["role"], "content": [{"type": "text", "text": c}]})
            continue
        parts = []
        for p in c:
            t = p["type"]
            if t == "text":
                parts.append({"type": "text", "text": p["text"]})
            elif t == "image_url":
                url = p["image_url"]["url"]
                if url.startswith("data:"):
                    parts.append({"type": "image", "url": save_tmp(url, ".jpg")})
                else:
                    parts.append({"type": "image", "url": url})
            elif t == "input_audio":
                fmt = p["input_audio"].get("format", "wav")
                parts.append({"type": "audio", "audio": save_tmp(p["input_audio"]["data"], "." + fmt)})
        out.append({"role": m["role"], "content": parts})
    return out

@app.get("/")
def health():
    return {"status": "ok", "model": MODEL_ID}

@app.post("/v1/chat/completions")
def chat(body: dict, authorization: str = Header(default="")):
    if API_KEY and authorization != f"Bearer {API_KEY}":
        raise HTTPException(401, "Invalid API key")
    msgs = convert(body["messages"])
    max_new = body.get("max_tokens", 256)
    temp = body.get("temperature", 0.7)
    with lock:
        inputs = processor.apply_chat_template(
            msgs, add_generation_prompt=True, tokenize=True,
            return_dict=True, return_tensors="pt").to(model.device)
        n = inputs["input_ids"].shape[-1]
        with torch.inference_mode():
            gen = model.generate(**inputs, max_new_tokens=max_new,
                                 do_sample=temp > 0, temperature=max(temp, 0.01))
        text = processor.decode(gen[0][n:], skip_special_tokens=True)
    return {
        "id": "chatcmpl-" + uuid.uuid4().hex[:8], "object": "chat.completion",
        "created": int(time.time()), "model": MODEL_ID,
        "choices": [{"index": 0, "finish_reason": "stop",
                     "message": {"role": "assistant", "content": text}}],
        "usage": {"prompt_tokens": n, "completion_tokens": gen.shape[-1] - n,
                  "total_tokens": gen.shape[-1]},
    }

Overwriting server.py


In [13]:
!ollama pull hf.co/unsloth/gemma-3n-E2B-it-GGUF:Q8_0

/bin/bash: line 1: ollama: command not found


In [17]:
import subprocess, os, time, requests
from kaggle_secrets import UserSecretsClient

env = os.environ.copy()
env["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
env["API_KEY"] = "APNI-SECRET-KEY"      
env["CUDA_VISIBLE_DEVICES"] = "0"

server = subprocess.Popen(
    ["python", "-m", "uvicorn", "server:app", "--host", "0.0.0.0", "--port", "8000"],
    env=env, stdout=open("server.log", "w"), stderr=subprocess.STDOUT)

for _ in range(60):                    
    try:
        print(requests.get("http://localhost:8000/").json()); break
    except Exception:
        time.sleep(10)
else:
    print(open("server.log").read()[-3000:])

{'status': 'ok', 'model': 'google/gemma-3n-E2B-it'}


In [18]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

import re
tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
                          stderr=subprocess.PIPE, text=True)
for line in tunnel.stderr:
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line)
    if m:
        print("Public URL:", m.group(0)); break

Public URL: https://measures-pdt-reaction-only.trycloudflare.com
